In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')


url = 'https://archive.ics.uci.edu/static/public/320/data.csv'
data = pd.read_csv(url)

print("Dataset shape:", data.shape)
print("\nFirst rows:\n", data.head())
print("\nTarget distribution:\n", data['G3'].describe())

def preprocess_data(df):
    df = df.copy()
    df['pass'] = (df['G3'] >= 10).astype(int)
    
    cat_cols = ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
    le_dict = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le
    
    print("Missing values:", df.isnull().sum().sum()) 
    return df, le_dict

data, le_dict = preprocess_data(data)
X = data.drop(['G1', 'G2', 'G3', 'pass'], axis=1, errors='ignore')  
y = data['pass']

print("\nFeatures shape:", X.shape)
print("Top features (correlations with pass):")
print(X.corrwith(y).abs().sort_values(ascending=False).head(10))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_train_scaled, y_train)
importances = pd.Series(rf_selector.feature_importances_, index=X.columns)
top_features = importances.nlargest(15).index  
print("\nTop 15 features by RF importance:")
print(top_features.tolist())

X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]
X_train_sel_scaled = scaler.fit_transform(X_train_sel)
X_test_sel_scaled = scaler.transform(X_test_sel)
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
}
estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
]
hybrid = StackingClassifier(estimators=estimators, final_estimator=MLPClassifier(hidden_layer_sizes=(50,), max_iter=300), cv=5)

models['Hybrid Stacking'] = hybrid
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nModel Comparison (Test Accuracy & CV Mean):")
for name, model in models.items():
    if name == 'Hybrid Stacking':
        scores_cv = cross_val_score(model, X_train_sel_scaled, y_train, cv=skf, scoring='accuracy')
        model.fit(X_train_sel_scaled, y_train)
        y_pred = model.predict(X_test_sel_scaled)
    else:
        scores_cv = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='accuracy')
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    
    acc_test = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'CV Mean': scores_cv.mean(), 'CV Std': scores_cv.std(), 'Test Acc': acc_test})
    
    print(f"{name}: CV {scores_cv.mean():.3f}±{scores_cv.std():.3f}, Test {acc_test:.3f}")
results_df = pd.DataFrame(results).sort_values('Test Acc', ascending=False)
print("\nRanked Results:\n", results_df.round(3))
best_model = max(results, key=lambda x: x['Test Acc'])['Model']
print(f"\nBest: {best_model}")
print("\nClassification Report for Best Model (Test Set):")
print(classification_report(y_test, y_pred))

Dataset shape: (649, 33)

First rows:
   school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        4   0  11  11  
1      5        3      3     1     1      3        2   9  11  11  
2      4        3      2     2     3      3        6  12  13  12  
3      3        2      2     1     1      5        0  14  14  14  
4      4        3      2     1     2      5        0  11  13  13  

[5 rows x 33 columns]

Target distribution:
 count    649.000000
mean  